
# Lab 3 — ROS Workspace, Packages, Publisher & Subscriber (C++ + Python)


## Overview
This lab teaches you how to build ROS software the “real” way:
1. Create a **workspace**
2. Create **packages**
3. Write and build **publisher/subscriber** nodes in **C++ (roscpp)** and **Python (rospy)**
4. Run them and **prove** they work using ROS inspection tools

> **Important:** Most code/commands in this notebook must be run in an Ubuntu **Terminal**, not inside Jupyter.

---

## Learning outcomes
By the end of this lab you can:
- Create and overlay a **catkin workspace**
- Create **catkin packages** and declare dependencies
- Implement a **publisher/subscriber** pair in **C++ (roscpp)** and **Python (rospy)**
- Build packages with `catkin_make` and manage dependencies with `rosdep`
- Verify your system with `rosnode`, `rostopic`, `rosmsg`, and `rqt_graph`

---

## Deliverables
Submit a single ZIP (or repo link) containing:
- Your workspace `ros_ws/` with both packages:
  - `cpp_pubsub_lab3`
  - `py_pubsub_lab3`
- Screenshots + short answers (Checklist at the end)



---
## Part A — Create and validate a catkin workspace

### A1) Create a new workspace
We’ll call it `ros_ws` (ROS workspace).

Run in Terminal:


In [ ]:

# Terminal
mkdir -p ~/ros_ws/src
cd ~/ros_ws
catkin_make



### A2) Overlay (source) the workspace
Sourcing makes your terminal aware of packages you build in this workspace.

Run:


In [ ]:

# Terminal (from ~/ros_ws)
source devel/setup.bash



### A3) Verify the overlay is active
Run:


In [ ]:

# Terminal
echo $ROS_PACKAGE_PATH



✅ **Checkpoint A:** You should see your workspace `~/ros_ws/src` listed *before* the system ROS path (e.g., `/opt/ros/melodic/share`).



---
## Part B — Create a C++ pub/sub package (roscpp)

### B1) Create the package
We’ll name it `cpp_pubsub_lab3` and depend on `roscpp` and `std_msgs`.

Run:


In [ ]:

# Terminal
cd ~/ros_ws/src
catkin_create_pkg cpp_pubsub_lab3 roscpp std_msgs



### B2) Create the C++ publisher (`talker.cpp`)
Create the source file:


In [ ]:

# Terminal
cd ~/ros_ws/src/cpp_pubsub_lab3/src
nano talker.cpp



Paste this **original** C++ publisher (written for this lab) into `talker.cpp`:


In [ ]:
cpp
#include <ros/ros.h>
#include <std_msgs/String.h>

#include <sstream>

int main(int argc, char** argv) {
  ros::init(argc, argv, "talker_cpp_lab3");
  ros::NodeHandle nh;

  ros::Publisher pub = nh.advertise<std_msgs::String>("chatter", 10);

  // Private parameter: ~publish_rate_hz (default 5.0)
  double publish_rate_hz = 5.0;
  nh.param("publish_rate_hz", publish_rate_hz, 5.0);

  ros::Rate rate(publish_rate_hz);
  int count = 0;

  while (ros::ok()) {
    std_msgs::String msg;
    std::stringstream ss;
    ss << "[cpp] lab3 hello #" << count;
    msg.data = ss.str();

    ROS_INFO("%s", msg.data.c_str());
    pub.publish(msg);

    ros::spinOnce();
    rate.sleep();
    ++count;
  }
  return 0;
}



### B3) Create the C++ subscriber (`listener.cpp`)
Create the file:


In [ ]:

# Terminal
cd ~/ros_ws/src/cpp_pubsub_lab3/src
nano listener.cpp



Paste this **original** C++ subscriber into `listener.cpp`:


In [ ]:
cpp
#include <ros/ros.h>
#include <std_msgs/String.h>

void chatterCallback(const std_msgs::String::ConstPtr& msg) {
  ROS_INFO("[cpp] I heard: %s", msg->data.c_str());
}

int main(int argc, char** argv) {
  ros::init(argc, argv, "listener_cpp_lab3");
  ros::NodeHandle nh;

  ros::Subscriber sub = nh.subscribe("chatter", 10, chatterCallback);
  ros::spin();
  return 0;
}



### B4) Wire the executables in `CMakeLists.txt`
Open `~/ros_ws/src/cpp_pubsub_lab3/CMakeLists.txt` and add these lines near the bottom (after `catkin_package(...)`):

```cmake
add_executable(talker_cpp src/talker.cpp)
add_executable(listener_cpp src/listener.cpp)

target_link_libraries(talker_cpp ${catkin_LIBRARIES})
target_link_libraries(listener_cpp ${catkin_LIBRARIES})

add_dependencies(talker_cpp ${catkin_EXPORTED_TARGETS})
add_dependencies(listener_cpp ${catkin_EXPORTED_TARGETS})
```

Open the file:


In [ ]:

# Terminal
nano ~/ros_ws/src/cpp_pubsub_lab3/CMakeLists.txt



### B5) Install missing dependencies (rosdep) and build
Run from the workspace root:


In [ ]:

# Terminal
cd ~/ros_ws
rosdep install -i --from-paths src --rosdistro melodic -y
catkin_make
source devel/setup.bash



### B6) Run and verify (C++)
Open terminals and run:

**Terminal 1:**


In [ ]:

roscore



**Terminal 2 (new terminal):**


In [ ]:

cd ~/ros_ws
source devel/setup.bash
rosrun cpp_pubsub_lab3 talker_cpp



**Terminal 3 (new terminal):**


In [ ]:

cd ~/ros_ws
source devel/setup.bash
rosrun cpp_pubsub_lab3 listener_cpp



✅ **Checkpoint B:** The listener prints messages that the talker publishes.

### B7) Inspect with ROS tools
In a new terminal:


In [ ]:

rosnode list
rostopic list
rostopic info /chatter
rostopic echo -n 1 /chatter
rosmsg show std_msgs/String



---
## Part C — Create a Python pub/sub package (rospy)

### C1) Create the package
Run:


In [ ]:

cd ~/ros_ws/src
catkin_create_pkg py_pubsub_lab3 rospy std_msgs



### C2) Create `scripts/` and the Python nodes
Run:


In [ ]:

cd ~/ros_ws/src/py_pubsub_lab3
mkdir -p scripts



Create the publisher script:


In [ ]:

nano ~/ros_ws/src/py_pubsub_lab3/scripts/talker.py
chmod +x ~/ros_ws/src/py_pubsub_lab3/scripts/talker.py



Paste this **original** Python publisher into `talker.py`:


In [ ]:
python
#!/usr/bin/env python
import rospy
from std_msgs.msg import String

def main():
    rospy.init_node("talker_py_lab3", anonymous=False)
    pub = rospy.Publisher("chatter", String, queue_size=10)

    publish_rate_hz = rospy.get_param("~publish_rate_hz", 5.0)
    rate = rospy.Rate(publish_rate_hz)

    count = 0
    while not rospy.is_shutdown():
        msg = String()
        msg.data = f"[py] lab3 hello #{count}"
        rospy.loginfo(msg.data)
        pub.publish(msg)
        count += 1
        rate.sleep()

if __name__ == "__main__":
    main()



Create the subscriber script:


In [ ]:

nano ~/ros_ws/src/py_pubsub_lab3/scripts/listener.py
chmod +x ~/ros_ws/src/py_pubsub_lab3/scripts/listener.py



Paste this **original** Python subscriber into `listener.py`:


In [ ]:
python
#!/usr/bin/env python
import rospy
from std_msgs.msg import String

def cb(msg):
    rospy.loginfo(f"[py] I heard: {msg.data}")

def main():
    rospy.init_node("listener_py_lab3", anonymous=False)
    rospy.Subscriber("chatter", String, cb, queue_size=10)
    rospy.spin()

if __name__ == "__main__":
    main()



### C3) Ensure scripts install correctly (CMakeLists)
Open `~/ros_ws/src/py_pubsub_lab3/CMakeLists.txt` and add this near the bottom:

```cmake
catkin_install_python(PROGRAMS
  scripts/talker.py
  scripts/listener.py
  DESTINATION ${CATKIN_PACKAGE_BIN_DESTINATION}
)
```

Open the file:


In [ ]:

nano ~/ros_ws/src/py_pubsub_lab3/CMakeLists.txt



### C4) Build and run (Python)
Build:


In [ ]:

cd ~/ros_ws
catkin_make
source devel/setup.bash



Run (in separate terminals; `roscore` must be running):

**Terminal 2:**


In [ ]:

cd ~/ros_ws
source devel/setup.bash
rosrun py_pubsub_lab3 talker.py



**Terminal 3:**


In [ ]:

cd ~/ros_ws
source devel/setup.bash
rosrun py_pubsub_lab3 listener.py



✅ **Checkpoint C:** The Python listener prints Python talker messages.

> If your C++ nodes are still running, they may also publish/subscribe to the same `/chatter` topic — that’s a valid demonstration that ROS is language-agnostic when types match.



---
## Part D — Engineering challenges

### Challenge 1 — Per-node publish rate parameter
1. Run the C++ talker at **2 Hz** using a private parameter.
2. Run the Python talker at **8 Hz** using a private parameter.
3. Show evidence using `rostopic hz /chatter`.



In [ ]:

# Examples
rosrun cpp_pubsub_lab3 talker_cpp _publish_rate_hz:=2.0
rosrun py_pubsub_lab3 talker.py _publish_rate_hz:=8.0

rostopic hz /chatter



### Challenge 2 — Topic remapping
1. Remap the C++ talker from `chatter` to `chatter_fast`.
2. Run a listener that subscribes to `chatter_fast`.
3. Provide `rostopic list` evidence showing both topics exist.



In [ ]:

rosrun cpp_pubsub_lab3 talker_cpp chatter:=chatter_fast
rosrun py_pubsub_lab3 listener.py chatter:=chatter_fast

rostopic list | grep chatter



### Challenge 3 — Make the message meaningful
Modify **one** talker so each message includes:
- a sequence number
- a timestamp (ROS time)
- your machine hostname

Then show a screenshot of the listener output.



---
## Submission checklist (what to submit)

1. Screenshot: terminal showing `catkin_make` success.
2. Screenshot: `rosnode list` showing your nodes.
3. Screenshot: `rostopic info /chatter` (or remapped topic) showing publisher/subscriber counts.
4. Evidence for each challenge (screenshots or copied terminal output).
5. Short answers (2–4 sentences each):
   - Why do we `source devel/setup.bash`?
   - What must match for different-language nodes to communicate on a topic?
   - When would you choose a **service** over a **topic**?



---
## Appendix — Ubuntu 24.04 track (ROS 2 Jazzy equivalents)

If you are on **Ubuntu 24.04**, use ROS 2 Jazzy and follow this equivalent workflow.

### 1) Create a workspace
```bash
mkdir -p ~/ros2_ws/src
cd ~/ros2_ws
colcon build
echo "source ~/ros2_ws/install/setup.bash" >> ~/.bashrc
source ~/.bashrc
```

### 2) Create packages
```bash
cd ~/ros2_ws/src
ros2 pkg create cpp_pubsub_lab3 --build-type ament_cmake --dependencies rclcpp std_msgs
ros2 pkg create py_pubsub_lab3 --build-type ament_python --dependencies rclpy std_msgs
```

### 3) Inspection tools
```bash
ros2 node list
ros2 topic list
ros2 topic info /chatter
ros2 topic echo --once /chatter
ros2 topic hz /chatter
```

### 4) Remapping
```bash
ros2 run py_pubsub_lab3 talker --ros-args -r chatter:=chatter_fast
ros2 run py_pubsub_lab3 listener --ros-args -r chatter:=chatter_fast
```
